# 21 — Hurtigtester av timing, generatorsamspill og reserveblend

**Henrik Remmen (541747), gruppe 119 — [119] SINTEF Databreach**

Notebook 19 / `hydro_lp_v6.csv` (**rapportert public 0.99571**) står urørt.
Notebook 20 ga svakere AUC med ukekorreksjon. Nå tester vi struktur som den
ukekorreksjonen ikke kunne representere.

**Kun lagrede valideringsprediksjoner fra 19. Ingen fulltrening, nye LP-løsninger eller submissions.**
Alle originale CSV-er er read-only; nye resultater havner i en unik mappe under
`artifacts/structure_probe_v7/`. Dette er en lokal forskningsnotebook, ikke professorleveransen.

## Research → fem små, faste tester

| Kandidat | Hypotese | Begrensning |
|---|---|---|
| local_smoothing | Svak 3-timers utjevning kan fjerne korte feil | Kan viske ut ekte overganger; glatting ble også prøvd i notebook 11 |
| soft_switch_chain | En svak start/stopp-straff kan gi mer sammenhengende planer | Ikke eksakt SHOP-fysikk; dimensjonsløs straff |
| plant_count_prior | Antall aktive generatorer på samme kraftverk er ikke uavhengig | Historiske frekvenser kan endres mellom år |
| plant_state_prior | Også generatorkombinasjonen kan gi ekstra informasjon | Mer risiko for å lære historiske ID-preferanser |
| conservative_v5_blend | 90 % LP + 10 % v5 kan fungere som reserve | Kan fortynne den sterke LP-modellen |

Plant-priorer er svært små frekvenstabeller, lært kun på foregående år etter purge.
Ingen generator tvinges ON/OFF; alle resultater forblir sannsynligheter.
Kjedemodellen bruker forward/backward-marginaler, ikke harde Viterbi-beslutninger.

Kilder:
- [SINTEF: generator, minimumseffekt og start/stopp](https://docs.shop.sintef.energy/objects/generator/generator.html)
- [SINTEF: kraftverksmodell](https://docs.shop.sintef.energy/objects/plant/plant.html)
- [Stanford CS262: kjedemodeller og forward/backward](https://web.stanford.edu/class/cs262/archives/notes/lecture9.pdf)
- [Scikit-learn: sannsynlighetsblending](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.VotingClassifier.html)

Detaljer, antakelser og avgrensninger står i `RESEARCH_STRUCTURE_PROBE_V7.md`.
Ingen eksterne data brukes.


In [1]:
# Første celle: ressurser og FROSNE hypoteser, ikke et parametersøk.
import os
CPU_THREADS = max(1, min(4, os.cpu_count() or 1))
for key in ("OMP_NUM_THREADS","OPENBLAS_NUM_THREADS","MKL_NUM_THREADS",
            "VECLIB_MAXIMUM_THREADS","NUMEXPR_NUM_THREADS"):
    os.environ[key] = str(CPU_THREADS)
CONFIG = dict(smooth_weight=.15, switch_penalty=.4, chain_blend=.25,
              prior_mix=.25, prior_strength=.25, v5_weight=.10)
MIN_DEVELOPMENT_GAIN = .00001   # Praktisk terskel, IKKE en signifikansgrense.
print(f"{CPU_THREADS} CPU-tråder; fem faste hurtigforsøk, ingen boosting.")


4 CPU-tråder; fem faste hurtigforsøk, ingen boosting.


## 1. Les data og kontroller opphav — ingen trening

Vi bruker samme kontrollerte innlasting som notebook 20: rådatafingeravtrykk,
case-ID-er, LP-cache og lagrede AUC-er må stemme med notebook 19.
Ingen notebook importeres eller kjøres; bare små hjelpefunksjoner brukes.

Tidsbruk er ikke benchmark-et på maskinen din; forsøket er ment å ta sekunder til få minutter.


In [2]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import sys
import time
import uuid
import importlib.metadata
import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/"src/sintef_structure_probe.py").is_file()
             and (p/"data/kernel").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Åpne notebooken inne i prosjektet med resultatene fra 19.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.sintef_weekly_probe import file_digest, load_probe_inputs, purged_training_mask
from src.sintef_structure_probe import candidates, evaluate_candidates, select_on_development

STARTED = time.perf_counter()
protected = sorted(set(list((ROOT/"data").rglob("*.csv"))
    + list((ROOT/"submissions").rglob("*.csv"))
    + list((ROOT/"artifacts/hydro_lp_v6").rglob("*.csv"))
    + list((ROOT/"artifacts/weekly_correction_probe").rglob("*.csv"))
    + list(ROOT.glob("*.csv"))))
ORIGINAL_HASHES = {str(p.relative_to(ROOT)):file_digest(p) for p in protected}
bundles, units, plants, topology, provenance = load_probe_inputs(ROOT)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")+"_"+uuid.uuid4().hex[:8]
OUT = ROOT/"artifacts/structure_probe_v7"/RUN_ID
OUT.mkdir(parents=True, exist_ok=False)
helper_paths = ["src/sintef_structure_probe.py", "src/sintef_weekly_probe.py"]
CODE_HASHES = {p:file_digest(ROOT/p) for p in helper_paths}
display(pd.DataFrame([dict(train_year=a,validation_year=b,
    train_weeks=int(purged_training_mask(bundles[a]["starts"],bundles[b]["starts"]).sum()),
    purged_weeks=int((~purged_training_mask(bundles[a]["starts"],bundles[b]["starts"])).sum()))
    for a,b in [(2020,2021),(2021,2022)]]))
print("Ingen originalfiler skrives. Nye resultater:", OUT)


,train_year,validation_year,train_weeks,purged_weeks
0,2020,2021,360,6
1,2021,2022,359,6


Ingen originalfiler skrives. Nye resultater: /Users/henrik/Documents/H26/Moderne Maksinlæring/sintef_project/artifacts/structure_probe_v7/20260928T234651Z_c52873a1


## 2. Utviklingssjekk: 2020 → 2021, velg før 2022

De label-lærte kraftverkspriorene tilpasses kun 2020 etter purge.
De andre korreksjonene er faste og trenger ingen tilpasning.
Én global kandidat velges etter 2021 micro-AUC hvis forbedringen overstiger terskelen.
Ingen vekter, generator-unntak eller kombinasjoner søkes.

De seks siste ukecasene i treningsåret fjernes før begge tidsdelinger.
2021 og 2022 er tidligere undersøkt: dette er utforskende, ikke et urørt holdout.


In [3]:
with threadpool_limits(limits=CPU_THREADS):
    development_predictions, development_priors, development_split = candidates(
        bundles[2020], bundles[2021], units, CONFIG)
    development_metrics, development_timing = evaluate_candidates(
        bundles[2021], development_predictions, units)
development_metrics.to_csv(OUT/"development_metrics_2021.csv", index=False)
development_timing.to_csv(OUT/"development_timing_2021.csv", index=False)
selection = select_on_development(development_metrics, MIN_DEVELOPMENT_GAIN)
selection.update(config=CONFIG.copy(), code_hashes=CODE_HASHES,
                 notebook19_identity=provenance["notebook19_identity"],
                 input_sha256=provenance["input_sha256"], split=development_split,
                 minimum_development_gain=MIN_DEVELOPMENT_GAIN)
(OUT/"selection_before_2022.json").write_text(json.dumps(selection,indent=2),encoding="utf-8")
display(development_metrics.query("group == 'all'").merge(development_timing,on="model")
        .sort_values("micro_auc",ascending=False).round(7))
print("Valgt FØR 2022:", selection["selected"])
print("Ikke endre CONFIG eller velg en ny metode etter neste celle og kall 2022 uavhengig.")


,model,group,label,micro_auc,brier,weekly_level_mae,within_week_auc,transition_brier,soft_temporal_variation
1,local_smoothing,all,all,0.993821,0.030790,0.035239,0.966516,0.183333,0.020175
2,soft_switch_chain,all,all,0.993816,0.031002,0.034942,0.966375,0.182306,0.020591
0,original_lp,all,all,0.993805,0.030839,0.035226,0.966262,0.179435,0.021060
4,plant_state_prior,all,all,0.993778,0.030856,0.035157,0.966339,0.179891,0.021055
3,plant_count_prior,all,all,0.993771,0.030867,0.035179,0.966331,0.179911,0.021062
5,conservative_v5_blend,all,all,0.993763,0.030599,0.036521,0.967080,0.176114,0.020880


Valgt FØR 2022: local_smoothing
Ikke endre CONFIG eller velg en ny metode etter neste celle og kall 2022 uavhengig.


## 3. Senere-årskontroll: 2021 → 2022

Denne cellen stopper hvis konfigurasjon eller hjelpekode er endret etter valget.
Kraftverkspriorene tilpasses på purget 2021, aldri på fasit fra 2022.
Alle kandidater vises for å lære av forsøket, men det **frosne valget** vurderes separat.
Eventuell 2022-vinner er ikke en automatisk erstatning for dette valget.


In [4]:
frozen = json.loads((OUT/"selection_before_2022.json").read_text())
current_code = {p:file_digest(ROOT/p) for p in helper_paths}
if (frozen["config"] != CONFIG or frozen["code_hashes"] != current_code
    or frozen["minimum_development_gain"] != MIN_DEVELOPMENT_GAIN
    or frozen["notebook19_identity"] != provenance["notebook19_identity"]):
    raise ValueError("Oppsettet er endret etter utviklingsvalget. Start et nytt, merket forsøk.")
with threadpool_limits(limits=CPU_THREADS):
    audit_predictions, audit_priors, audit_split = candidates(
        bundles[2021], bundles[2022], units, CONFIG)
    audit_metrics, audit_timing = evaluate_candidates(bundles[2022], audit_predictions, units)
audit_metrics.to_csv(OUT/"audit_metrics_2022.csv", index=False)
audit_timing.to_csv(OUT/"audit_timing_2022.csv", index=False)
np.savez_compressed(OUT/"validation_predictions_2022.npz",
                    case_ids=bundles[2022]["case_ids"], **audit_predictions)
display(audit_metrics.query("group == 'all'").merge(audit_timing,on="model")
        .sort_values("micro_auc",ascending=False).round(7))
print("2022-priorenes treningssplit:", audit_split)


,model,group,label,micro_auc,brier,weekly_level_mae,within_week_auc,transition_brier,soft_temporal_variation
5,conservative_v5_blend,all,all,0.992880,0.030741,0.033486,0.976374,0.170269,0.028643
4,plant_state_prior,all,all,0.992578,0.031502,0.033156,0.976139,0.174647,0.028716
3,plant_count_prior,all,all,0.992569,0.031523,0.033202,0.976139,0.174704,0.028723
2,soft_switch_chain,all,all,0.992565,0.031529,0.033013,0.976157,0.176086,0.028246
0,original_lp,all,all,0.992564,0.031349,0.033044,0.976117,0.173574,0.028700
1,local_smoothing,all,all,0.992563,0.031280,0.033031,0.976261,0.177008,0.027684


2022-priorenes treningssplit: {'training_weeks': 359, 'purged_weeks': 6}


## 4. Sammenligning og timing — ingen nye modeller

- **Micro-AUC:** hovedmål. Høyere er bedre.
- **Within-week AUC:** rangering av timer innen samme uke/generator med begge klasser;
  vektet etter antall positive × negative par. Konstante uker gir ingen «gratis» poeng.
- **Transition Brier:** feil ved faktiske ON/OFF-overganger, kun til diagnostikk.
- **Soft temporal variation:** hvor brå sannsynlighetene er. Lavere er IKKE automatisk bedre.
- **Weekly level MAE:** nivåfeil; lavere er bedre, men den kan være i konflikt med AUC.

Ingen hard avrunding og ingen tvungen enighet mellom ulike ukecase som overlapper i kalenderen.


In [5]:
comparisons = []
for year, table in [(2021,development_metrics),(2022,audit_metrics)]:
    overall = table.query("group == 'all'").set_index("model")
    reference = float(overall.loc["original_lp","micro_auc"])
    for model,row in overall.iterrows():
        comparisons.append(dict(year=year,model=model,micro_auc=float(row.micro_auc),
                                delta_vs_lp=float(row.micro_auc-reference)))
comparison = pd.DataFrame(comparisons)
comparison.to_csv(OUT/"comparisons.csv",index=False)
display(comparison.pivot(index="model",columns="year",values=["micro_auc","delta_vs_lp"]).round(7))

chosen = frozen["selected"]
delta = float(comparison.query("year == 2022 and model == @chosen").delta_vs_lp.iloc[0])
if chosen == "original_lp":
    verdict = "BEHOLD LP: ingen utviklingskandidat passerte den faste terskelen."
elif delta <= 0:
    verdict = "IKKE BEKREFTET: kandidaten valgt på 2021 forbedret ikke 2022. Behold LP."
else:
    verdict = "LOVENDE PUNKTESTIMAT: valgt på 2021 og positiv på 2022; ikke signifikans eller ny submission."
print("Frosset kandidat:", chosen, "| 2022 delta:", f"{delta:+.7f}")
print(verdict)
for group in ("unit","month"):
    subset = audit_metrics.query("group == @group and model in ['original_lp', @chosen]")
    detail = subset.pivot(index="label",columns="model",values="micro_auc")
    if chosen != "original_lp":
        detail["delta"] = detail[chosen]-detail["original_lp"]
        detail = detail.sort_values("delta")
    display(detail.round(7))
print("Ikke plukk generator-spesifikke vinnere fra denne tabellen uten ny, separat validering.")


micro_auc           delta_vs_lp              
year                       2021      2022        2021          2022
model                                                              
conservative_v5_blend  0.993763  0.992880   -0.000042  3.160000e-04
local_smoothing        0.993821  0.992563    0.000016 -8.000000e-07
original_lp            0.993805  0.992564    0.000000  0.000000e+00
plant_count_prior      0.993771  0.992569   -0.000034  5.200000e-06
plant_state_prior      0.993778  0.992578   -0.000026  1.380000e-05
soft_switch_chain      0.993816  0.992565    0.000012  1.300000e-06

Frosset kandidat: local_smoothing | 2022 delta: -0.0000008
IKKE BEKREFTET: kandidaten valgt på 2021 forbedret ikke 2022. Behold LP.


model,local_smoothing,original_lp,delta
label,,,
Kjela_G1,0.987788,0.987871,-8.320000e-05
Hogga_G1,0.983770,0.983808,-3.840000e-05
Vesle Kjela_G1,0.995713,0.995724,-1.140000e-05
Lio_G1,0.975713,0.975722,-8.900000e-06
Vinje_G1,0.993472,0.993474,-2.300000e-06
Tokke_G3,0.997978,0.997977,3.000000e-07
Byrte_G1,0.999508,0.999507,1.200000e-06
Tokke_G2,0.996290,0.996289,1.400000e-06
Tokke_G1,0.998044,0.998041,3.300000e-06


model,local_smoothing,original_lp,delta
label,,,
9,0.997725,0.997764,-0.000039
3,0.995586,0.995611,-0.000025
2,0.996923,0.996932,-0.000009
1,0.993811,0.993810,0.000002
4,0.961377,0.961373,0.000004
12,0.997783,0.997776,0.000007
8,0.997268,0.997259,0.000009
11,0.993659,0.993636,0.000024
5,0.990794,0.990753,0.000040


Ikke plukk generator-spesifikke vinnere fra denne tabellen uten ny, separat validering.


## 5. Beskytt originalene og lagre sporbarhet

Ingen Kaggle-CSV produseres. Vi vil først se om noe hjelper utover original LP.
En eventuell innlevering trenger senere en selvstendig kortnotebook som gjenskaper både
grunnmodellen og etterbehandlingen fra rådata. Denne forskningsnotebooken gjenbruker artifacts.


In [6]:
changed = [p for p,h in ORIGINAL_HASHES.items()
           if not (ROOT/p).is_file() or file_digest(ROOT/p) != h]
if changed:
    raise RuntimeError(f"Originale CSV-er endret siden start, eventuelt av annen prosess: {changed}")
prior_rows = []
for year,prior_map in [(2021,development_priors),(2022,audit_priors)]:
    for plant,prior in prior_map.items():
        for state,ce,se in zip(prior["states"],prior["count_effect"],prior["state_effect"]):
            prior_rows.append(dict(validation_year=year,plant=plant,
                units="|".join(units[i] for i in prior["ids"]),
                state="".join(str(int(v)) for v in state), count_effect=float(ce),state_effect=float(se)))
pd.DataFrame(prior_rows).to_csv(OUT/"fitted_prior_diagnostics.csv",index=False)
nb_path = ROOT/"notebooks/21_structure_and_timing_probe.ipynb"
sources = [c["source"] for c in json.loads(nb_path.read_text())["cells"]]
manifest = dict(run_id=RUN_ID,config=CONFIG,selection=frozen,verdict=verdict,audit_delta=delta,
    provenance=provenance,helper_sha256=current_code,
    notebook_sources_sha256=hashlib.sha256(json.dumps(sources).encode()).hexdigest(),
    packages={p:importlib.metadata.version(p) for p in ("numpy","pandas","scipy","scikit-learn","threadpoolctl")},
    threads=CPU_THREADS,original_csv_sha256=ORIGINAL_HASHES,original_csv_unchanged=True,
    elapsed_seconds=round(time.perf_counter()-STARTED,2))
(OUT/"probe_manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")
print(f"Ferdig. {len(ORIGINAL_HASHES)} originale CSV-er kontrollert uendret.")
print("Sekunder:", manifest["elapsed_seconds"])
print("Resultater:",OUT)
print("Lagre notebooken (Cmd+S). Da kan resultatene også leses direkte fra notebookfilen.")


Ferdig. 253 originale CSV-er kontrollert uendret.
Sekunder: 5.89
Resultater: /Users/henrik/Documents/H26/Moderne Maksinlæring/sintef_project/artifacts/structure_probe_v7/20260928T234651Z_c52873a1
Lagre notebooken (Cmd+S). Da kan resultatene også leses direkte fra notebookfilen.
